# P04 · Notebook 35 · Test 3 (DCASE 2025 Task 2): pooling the expert, and a score bank for a self-weighted fusion

**Where we stand.** On Test 3 (eight machine types never heard) our best version is the expert trained with attribute classes: 62.28 %. The first team of the challenge had 61.63 %. Two 2026 papers reach more **without any training**: 62.4 % (frozen BEATs with a different pooling, arXiv 2603.04605) and 63.95 % (several frozen encoders, weighted without labels, arXiv 2609.27362).

**What this notebook does. No training.**
- It reads the audio and the models with notebook 30's own cells, unchanged.
- It reads notebook 30's six trainings with attribute classes from Drive › `P04_FirstShot` (it does not change that folder).
- Every clip goes through the two frozen BEATs models and through the six trained ones, and its tokens are pooled in eleven ways (mean, max, GeM, relative-deviation pooling, and both together).
- It scores every test clip under 122 candidates, and saves a small "score bank" (also the scores of the reference clips and of pseudo-outliers, which a fusion needs to weight itself without labels).
- **It does not read the evaluation labels.** It prints the development table only. The evaluation numbers and the fusions are computed from the bank afterwards, by the rules written first.

**Rules** written before this notebook existed: `results/mimii_beats/RULES_pooling_first_shot_2025_written_before_running.md`.

**Run:**
- Runtime › Change runtime type › a GPU (T4 is enough; a faster one is quicker). Then Runtime › Run all.
- Drive must hold `P04_BEATs` (both BEATs files) and `P04_FirstShot` (notebook 30's folder), as they are now.
- **Time, estimated:** downloads and the audio stores about 15 minutes; eight passes over 18,000 clips, about 7 minutes each on a T4 (notebook 24 measured 6.5 minutes per pass on a T4); the bank a few minutes. **About 1.5 hours on a T4**, well under half an hour on the G4.
- Each finished pass is saved in Drive › `P04_FirstShot35` › `pooled` (about 3 GB in all; you can delete that folder afterwards). If Colab disconnects, run all again: finished passes are skipped.

**Licences:** DCASE 2025 Task 2 data: CC BY 4.0. BEATs: MIT. ECHO: as on its model card.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, base64, importlib
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_FirstShot35", Path("/content/work"), MY / "P04_BEATs"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_FirstShot35")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", "P04_BEATs"))
if not CKPT_DIR.exists():
    raise SystemExit(f"missing {CKPT_DIR}: the folder P04_BEATs (with BEATs_iter3_plus_AS2M.pt, as notebook 1 left it) must be in Google Drive › MyDrive")
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_first_shot_2025_written_before_running.md"
# DCASE 2025 Task 2 on Zenodo: development, additional training and evaluation (MD5s from the record pages)
ZD, ZX, ZE = "https://zenodo.org/records/15097779/files/", "https://zenodo.org/records/15392814/files/", "https://zenodo.org/records/15519362/files/"
TYPES_DEV = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]
TYPES_EVAL = ["AutoTrash", "BandSealer", "CoffeeGrinder", "HomeCamera", "Polisher", "ScrewFeeder", "ToyPet", "ToyRCCar"]
DMD5 = {"ToyCar": "16bd2069065281ea7ac0e96894f8592c", "ToyTrain": "c2524ff503bc1631ca308dc840bc5010", "bearing": "b3a13693b69b477ce814db8b40783955",
        "fan": "61b6aae6d5ebb8cb937faa86741fbf47", "gearbox": "9542948d980fd575426f60f2a41060b8", "slider": "6c1f2d2ef0bc1a960db5938adb8425d1",
        "valve": "5640c0dc03982107ef354fe9480c9a83"}
XMD5 = {"AutoTrash": "0d89113bee9837402c609cb466c4a953", "BandSealer": "0e43610e140e34e0187b25f09d1f447b", "CoffeeGrinder": "9f39058361fde38fd5e24a52262d5977",
        "HomeCamera": "f81a89d772b1454af476059cb9800788", "Polisher": "09e62aea7f2a82b417e7c388c1060e92", "ScrewFeeder": "4a30f5aa52f019cac1d8653931ac4f44",
        "ToyPet": "0684d7185b4feea26b8479324f9c6573", "ToyRCCar": "74dc1d920983255d53b5214b0429b4d8"}
EMD5 = {"AutoTrash": "b33216913b71328d813bcbb3ec7415d7", "BandSealer": "78a582ca8092b35861542b1c34268952", "CoffeeGrinder": "359951de42956ef9a853dec6121a9517",
        "HomeCamera": "1b2e09f3283be0531fafeb5a66b58dd4", "Polisher": "7e50e4296cd7bd0ab94df37673df313e", "ScrewFeeder": "88035983d52753cf0bf2c9a692740222",
        "ToyPet": "c38c60b36b3cfdff5429fb6673eb9c57", "ToyRCCar": "1a6797436fe20d1a86224e65c8c19563"}
DC_DEV = {t_: (ZD + f"dev_{t_}.zip?download=1", f"dev_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_{t_}_train.zip?download=1", f"eval_data_{t_}_train.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_{t_}_test.zip?download=1", f"eval_data_{t_}_test.zip", h) for t_, h in EMD5.items()}
# the evaluation set's ground truth and the official evaluator: one repository at one commit. EV_SHA is the SHA-256 of the script and its
# two tool files together; GT_SHA of the 24 ground-truth files together (each file's path, a zero byte and its content, in the order below)
EV_COMMIT = "f903124a40a4cdb3faf2b2d6fa69c6a9f1794192"
EV_ARCHIVE = (f"https://github.com/nttcslab/dcase2025_task2_evaluator/archive/{EV_COMMIT}.zip", "dcase2025_task2_evaluator.zip")
EV_SCRIPT, EV_SHA = "dcase2025_task2_evaluator.py", "b20577511228d4243264fb339c06f86d2766ec50334609a3a45c335cc3b11102"
EV_TOOLS = ["tools/plot_anm_score.py", "tools/test_plots.py"]
GT_PARTS = ("ground_truth_data", "ground_truth_domain", "ground_truth_attributes")   # labels, domains, and the name table the evaluator needs
GT_SHA = "e54a474722f1fb5d878bed257c75444251eeefae4e4dd40a0f5363f8cec68850"
SECTION = "section_00"                                                # every type has one section
SR, LDN_FLOOR = 16000, 1e-6
NS_RULE = 10 * SR                                                     # every system reads 10 s of a clip: the first 10 s, or a shorter clip repeated to fill them
NS_MIN = 1 * SR                                                       # gate G0(b): a clip is at least 1 s
# Part 1 · the frozen recipe: notebook 23's lines, unchanged (notebook 21's settings; the track list fixes the random streams)
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS) + ["DC"]                                            # fixed order for random seeds
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)        # notebook 21's, unchanged
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
B3_SHA_N22B = "ba0f139a46b3a22f4ff78873db92214146cddfaad12db7c026f6dd758df1be6a"
ENCODERS = {"BEATs": {"name": "BEATs iter3+ (AS2M)", "dim": 768, "layers": 12},
            "BEATs-iter3": {"name": "BEATs iter3", "dim": 768, "layers": 12, "file": "BEATs_iter3.pt",
                            "sed_file": "BEATs_iter3_from_PretrainedSED.pt",          # Amendment 1: the third source
                            "sed": "https://github.com/fschmid56/PretrainedSED/releases/download/v0.0.1/BEATs_ssl.pt",
                            "share": "https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5",
                            "azure": ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt?sv=2020-08-04"
                                      "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")},
            "EAT-base": {"name": "EAT-base (epoch 30, pre-trained)", "dim": 768, "layers": 12, "repo": "worstchan/EAT-base_epoch30_pretrain"},
            "EAT-large": {"name": "EAT-large (epoch 20, pre-trained)", "dim": 1024, "layers": 24, "repo": "worstchan/EAT-large_epoch20_pretrain"}}
NEW_ENCODERS = ["BEATs-iter3"]                                       # notebook 23 needs only BEATs iter3 (MIC+ and MIC)
ENC_ORDER = list(ENCODERS)
EAT_NORM, EAT_FRAMES = (-4.268, 4.569), 1024                         # EAT's model card
GB_RATIO = 1.5
FROZEN = [("BEATs", "B", [0, 1, 2]), ("BEATs-iter3", "B3", [0, 1, 2])]   # MIC's final system: encoder, file code, seeds
# Part 2 · the second view: notebook 25's lines, unchanged
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES = 313, (8, 20), 15                # the reference's sizes for 10 s; a type is a machine: 15 classes
CAL_MIN = 30                                                          # a spread of form ZP needs at least 30 clips per type
MIN_SOURCE_FIRST = 600                                                # gate G0(b): a type's first 600 training clips by name are source-domain clips
GDEV_BAR = 0.55                                                       # gate G-DEV, on the development set's source-domain line: a system at chance has about 0.50
GDEV_SYSTEMS = ["ZP"]                                                 # the gate reads the system as it is; its two parts are reported beside it
AS_IS = ["ZP", "frozen", "view_paper"]                                # the system as it is and its two parts
ADOPT_BAR = 0.01                                                      # the choice: attribute classes are the result from this much over the system as it is
# the yardsticks: the official score (%) of the best system of each of the challenge's 35 teams (the results page, read twice)
TEAMS = {'Wang_MYPS_task2_3': 61.62755928284949, 'Saengthong_SCITOK_task2_2': 61.569433614253896, 'Yang_NBU_task2_1': 61.20134568401735,
         'Fujimura_NU_task2_1': 59.99466314801132, 'Jiang_THUEE_task2_2': 59.79329753333621, 'Zheng_SJTU-AITHU_task2_2': 59.49972264254595,
         'Guan_HEU_task2_1': 58.25344060002535, 'Ozeki_MELCO_task2_1': 58.23359688192865, 'Zhou_XAUAT_task2_1': 58.1881454965969,
         'Huang_XJU_task2_1': 58.14052430128241, 'Fu_CUMT_task2_4': 58.13668049404887, 'Qian_nivic_task2_2': 58.01974104378345,
         'Kim_AISTAT_task2_4': 57.95495072758183, 'Zhong_USTC_task2_4': 57.874646573614505, 'Wang_UniS_task2_1': 57.754303274825524,
         'Cai_NCUT_task2_3': 57.5585903081211, 'WT_IACAS_task2_2': 57.44073696264025, 'CHUNG_KUCAU_task2_4': 57.227161466047036,
         'Wang_ZJU_task2_4': 56.866003688314095, 'Sera_TMU_task2_1': 56.567561042877735, 'Ding_HFUU_task2_4': 56.37774443771024,
         'Zhao_CUMT_task2_3': 56.05710040362136, 'Zhang_DKU_task2_4': 55.76153468345078, 'Zhang_NWPU_task2_2': 55.42463764307989,
         'Lin_IASP_task2_4': 54.776114357402165, 'Li_XJTLU_task2_2': 53.469426047012206, 'Zhou_XJU_task2_4': 52.90746388052138,
         'Lobanov_ITMO_task2_2': 52.52595001212435, 'Bian_TGU_task2_3': 51.417480569726926, 'Chao_BUCT_task2_3': 50.13154697532129,
         'Vijayyan_SNUC_task2_1': 49.90505686153821, 'Kim_DAU_task2_2': 48.96578012474582, 'Dung_CNTT1PTIT_task2_1': 48.71892564275941,
         'Kret_CU_task2_1': 47.90299388774425, 'Emon_HDK_task2_1': 45.15099353175931}
BASELINES = {"DCASE2025_baseline_task2_MAHALA": 56.50558189601554, "DCASE2025_baseline_task2_MSE": 54.42827820713174}
# the result's words, on scores in % of two decimals: above the first team; at least the 3rd team's; at least the 18th team's (the middle of 35);
# at least the better baseline's
Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE = 61.63, 61.20, 57.23, 56.51
Q3_BAR = 0.01                                                         # the fusion's words, on the official score
AE_EVAL_SHIFT = (71.99, 47.88)                                        # the Mahalanobis baseline on the evaluation set: harmonic mean AUC (source), AUC (target)
# the baselines' development tables (the task page): per type AUC (source), AUC (target), pAUC, %
PUBLISHED_DEV = {"Autoencoder baseline (MSE)": {"ToyCar": (71.05, 53.52, 49.7), "ToyTrain": (61.76, 56.46, 50.19), "bearing": (66.53, 53.15, 61.12),
                                                "fan": (70.96, 38.75, 49.46), "gearbox": (64.8, 50.49, 52.49), "slider": (70.1, 48.77, 52.32),
                                                "valve": (63.53, 67.18, 57.35)},
                 "Autoencoder baseline (Mahalanobis)": {"ToyCar": (73.17, 50.91, 49.05), "ToyTrain": (50.87, 46.15, 48.32), "bearing": (63.63, 59.03, 61.86),
                                                        "fan": (77.99, 38.56, 50.82), "gearbox": (73.26, 51.61, 55.07), "slider": (73.79, 50.27, 53.61),
                                                        "valve": (56.22, 61.0, 52.53)}}
NAME = {t_: t_ for t_ in TYPES_DEV + TYPES_EVAL}; NAME["slider"] = "slide rail"
EVAL_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 1e-9, 3.0, 25.0
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"])
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    TYPES_DEV, TYPES_EVAL = list(_dc["dev"]), list(_dc["add"])        # the synthetic sets hold fewer machine types
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items()} for k_ in ("dev", "add", "eval"))
    EV_ARCHIVE, EV_SHA, GT_SHA = tuple(_dc["evaluator_archive"]), _dc["evaluator_sha256"], _dc["ground_truth_sha256"]
    CFG.update(_c.get("cfg", {})); SV.update(_c.get("sv", {}))
    for k_, v_ in _c.get("encoders", {}).items():
        ENCODERS[k_].update(v_)
    CAL_MIN, NS_RULE, MIN_SOURCE_FIRST = _c.get("cal_min", CAL_MIN), _c.get("clip_samples", NS_RULE), _c.get("min_source_first", MIN_SOURCE_FIRST)
    GDEV_BAR, REF_CLASSES = _c.get("gdev_bar", GDEV_BAR), len(TYPES_DEV) + len(TYPES_EVAL)
TYPES = TYPES_DEV + TYPES_EVAL
DEV_IDS = {t_: ([SECTION] if t_ in TYPES_DEV else []) for t_ in TYPES}   # the sections with labelled test clips
EVAL_IDS = {t_: ([SECTION] if t_ in TYPES_EVAL else []) for t_ in TYPES}  # the sections with unlabelled test clips
assert sorted(DC_DEV) == sorted(TYPES_DEV) and sorted(XZIPS) == sorted(DC_EVAL) == sorted(TYPES_EVAL) and not set(TYPES_DEV) & set(TYPES_EVAL)
FROZEN_CAP = CFG["xc_cap"]                                            # the frozen recipe trains on the first 600 training clips of a type
assert [sd for _, _, sd in FROZEN] == [CFG["se_seeds"]] * 2 and ENC_ORDER[:2] == [e for e, _, _ in FROZEN]
assert len(TEAMS) == 35 and sorted(TEAMS.values(), reverse=True) == list(TEAMS.values()), "the team ranking is not in order"
_ts = [round(v, 2) for v in TEAMS.values()]
assert (_ts[0], _ts[2], _ts[17], round(BASELINES["DCASE2025_baseline_task2_MAHALA"], 2)) == (Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE), "the words' lines are not the table's"
print("rules:", RULES, "·", json.dumps(CFG), "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")
if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
    raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")

In [ ]:
# 3 · The official BEATs code (notebook 23's cell, unchanged: the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 4 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 5 · Model pieces (notebook 23's cell, unchanged; notebook 2's code): our forward pass, LoRA, the head and the sub-center ArcFace layer
def encode(x, amp):
    """Waveforms (B, N) in [-1, 1] → mean over time of the last layer (B, 768), float32.
    Same steps as BEATs.extract_features without padding; the filter banks are always computed in float32."""
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # (B, frames, 128)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1)


class LoRALinear(nn.Module):
    """y = W x + (α / r) · B A x. W stays frozen; B starts at zero, so the model starts exactly as the frozen one."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(base.out_features, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class Head(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bn = nn.BatchNorm1d(d_out)

    def forward(self, x):
        return self.bn(self.lin(x))


class SubCenterArcFace(nn.Module):
    """Cosine to the closest of k centres per class; additive angular margin m on the true class; scale s."""
    def __init__(self, d, n_classes, k, s, m):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes * k, d))
        nn.init.xavier_uniform_(self.W)
        self.n, self.k, self.s, self.m = n_classes, k, s, m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def cosine(self, e):
        c = F.linear(F.normalize(e.float(), dim=1), F.normalize(self.W.float(), dim=1))
        return c.view(-1, self.n, self.k).max(dim=2).values

    def loss(self, cos, y):
        sin = torch.sqrt((1.0 - cos * cos).clamp(0.0, 1.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)      # keeps the target logit monotonic near θ = π
        one = F.one_hot(y, self.n).to(cos.dtype)
        return F.cross_entropy(self.s * (one * phi + (1.0 - one) * cos), y)


def equal_length_batches(clips, idx, batch):
    """Consecutive clips of the same length, at most `batch` per group (the same grouping as notebook 1)."""
    buf = []
    for i in idx:
        if buf and len(clips[i]) != len(clips[buf[0]]):
            yield buf; buf = []
        buf.append(i)
        if len(buf) == batch:
            yield buf; buf = []
    if buf:
        yield buf


def as_float(clips, ids):
    return torch.from_numpy(np.stack([clips[i] for i in ids]).astype(np.float32) / 32768.0).to(DEVICE)

In [ ]:
# 6 · LoRA on the query and value projections of all 12 layers, with one switch (PD's frozen teacher runs with it off); checks; a timed step
for p_ in model.parameters():
    p_.requires_grad_(False)                                          # the original weights never change


class LoRASwitch(LoRALinear):
    on = True                                                         # one switch for every block

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale if LoRASwitch.on else y


def tokens(x, amp):
    # waveforms (B, N) in [-1, 1] → last-layer tokens (B, T, 768), float32: notebook 2's encode without the final mean
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float()


rng0 = np.random.default_rng(0)
CROP = int(CFG["crop_s"] * SR)
x4 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (4, CROP)).astype(np.float32)).to(DEVICE)
with torch.no_grad():
    Z4 = tokens(x4, amp=False).mean(1)
    ref = model.extract_features(x4)[0].mean(1)
    gap_e = float((encode(x4, amp=False) - Z4).abs().max())
gap = float((Z4 - ref).abs().max())
print(f"our forward pass against BEATs.extract_features: largest difference {gap:.2e} (notebook 2's encode: {gap_e:.1e})")
assert gap < 1e-3 and gap_e < 1e-5, "stop: our forward pass does not reproduce BEATs"
LORA = []
for layer in model.encoder.layers:
    att = layer.self_attn
    for name in ("q_proj", "v_proj"):
        w = LoRASwitch(getattr(att, name), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"])
        setattr(att, name, w); LORA.append(w)
with torch.no_grad():
    gap = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
print(len(LORA), "LoRA blocks · with LoRA at zero, largest difference from frozen BEATs:", f"{gap:.2e}")
assert gap < 1e-5, "stop: adding LoRA changed the model before training"


def lora_params():
    return [p for w in LORA for p in (w.A, w.B)]


def fresh_adapters(seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen BEATs
    torch.manual_seed(seed)
    for w in LORA:
        w.reset()
    head = Head(768, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


head_t, arc_t = fresh_adapters(12345, 8)
opt_t = torch.optim.AdamW(lora_params() + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
model.train(); head_t.train(); arc_t.train(); times = []
for k in range(3):
    xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
    yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    loss = arc_t.loss(arc_t.cosine(head_t(tokens(xb, amp=AMP).mean(1))), yb)
    opt_t.zero_grad(set_to_none=True); loss.backward()
    if k == 0:
        assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in LORA if w.B.grad is not None) > 0, "stop: no gradient reaches LoRA"
    opt_t.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
model.eval()
with torch.no_grad():
    moved = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = False
    off = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = True
print(f"after 3 steps: with LoRA on the output moved by {moved:.2e}; with the switch off it differs from frozen BEATs by {off:.2e}")
assert moved > 1e-6 and off < 1e-5, "stop: the LoRA switch does not work"
del head_t, arc_t, opt_t
for w in LORA:
    w.reset()
STEP_S = min(times[1:])
print(f"BEATs iter3+ (AS2M): one training step {STEP_S:.3f} s")

In [ ]:
# 7 · BEATs iter3 (notebook 23's cell, unchanged; notebook 22b's loader and file): load, smoke test, LoRA, GB, a timed step; gate G0(h)
import torchaudio.compliance.kaldi as ta_kaldi


def beats_fb_tokens(m, fb, amp):
    # filter banks (B, frames, 128) of any BEATs model → last-layer tokens (B, T·8, 768): the forward pass of cell 9
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = m.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = m.layer_norm(f)
        if m.post_extract_proj is not None:
            f = m.post_extract_proj(f)
        h, _ = m.encoder(m.dropout_input(f), padding_mask=None)
    return h.float()


def kaldi_frames(n):
    # frames of a 25 ms / 10 ms Kaldi filter bank of n samples (edges snipped)
    return 1 + (n - 400) // 160


class LoRAQKV(nn.Module):
    """A fused query-key-value projection with LoRA on the query and value parts only: y = W x + (α / r) · [B_q A_q x, 0, B_v A_v x].
    B starts at zero, so the model starts exactly as the frozen one; LoRASwitch.on switches every block at once."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        d = base.in_features
        assert base.out_features == 3 * d, (base.in_features, base.out_features)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(2, r, d, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(2, d, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        with torch.no_grad():
            for i in range(2):
                nn.init.kaiming_uniform_(self.A[i], a=math.sqrt(5))
            self.B.zero_()

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        y = self.base(x)
        if not LoRASwitch.on:
            return y
        h = self.drop(x)
        dq = F.linear(F.linear(h, self.A[0]), self.B[0]) * self.scale
        dv = F.linear(F.linear(h, self.A[1]), self.B[1]) * self.scale
        return y + torch.cat([dq, torch.zeros_like(dq), dv], -1)


class Enc:
    # one encoder: waveforms → last-layer patch tokens (no CLS), its LoRA blocks, its patch layout, and pooling over the time
    # patches that start inside the clip
    def __init__(self, key, net, lora, info=None):
        self.key, self.name, self.dim, self.net, self.lora = key, ENCODERS[key]["name"], ENCODERS[key]["dim"], net, list(lora)
        self.layout, self.ok_lora, self.step_s, self.info = "frequency fastest", bool(lora), None, dict(info or {})

    def tokens(self, x, amp):
        with torch.autocast(device_type=DEVICE, enabled=False):
            fb = self.wave2fb(x.float())                              # filter banks always in float32
        return self.fb2tok(fb, amp)

    def grid(self, h, n_samples):
        # (B, L, D) tokens → (B, T, 8, D): the time patches that start inside a clip of n_samples, in this encoder's layout
        B, Lt, D = h.shape; Tp = Lt // 8
        g = h.view(B, Tp, 8, D) if self.layout == "frequency fastest" else h.view(B, 8, Tp, D).transpose(1, 2)
        return g[:, :max(1, min(Tp, -(-kaldi_frames(n_samples) // 16)))]

    def pool(self, h, n_samples):
        return self.grid(h, n_samples).mean((1, 2))

    def train(self):
        self.net.train()

    def eval(self):
        self.net.eval()

    def reset_lora(self):
        for w in self.lora:
            w.reset()


class BeatsEnc(Enc):
    def wave2fb(self, x):
        return self.net.preprocess(x)

    def fb2tok(self, fb, amp):
        return beats_fb_tokens(self.net, fb, amp)


class EatEnc(Enc):
    varlen = False

    def wave2fb(self, x):
        # EAT's model card: waveform minus its mean; Kaldi filter banks (128 mel, Hanning, 10 ms shift, no dither, HTK-compatible);
        # zero-padded to 1,024 frames (training crops: to a multiple of 16 frames when the model accepts it); then normalised
        out = []
        for w in x:
            w = w - w.mean()
            out.append(ta_kaldi.fbank(w.unsqueeze(0), htk_compat=True, sample_frequency=SR, use_energy=False, window_type="hanning",
                                      num_mel_bins=128, dither=0.0, frame_shift=10))
        fb = torch.stack(out); n = fb.shape[1]
        tgt = EAT_FRAMES if (n > EAT_FRAMES - 128 or not self.varlen) else 16 * math.ceil(n / 16)
        fb = F.pad(fb, (0, 0, 0, tgt - n)) if tgt >= n else fb[:, :tgt]
        return (fb - EAT_NORM[0]) / (2.0 * EAT_NORM[1])

    def fb2tok(self, fb, amp):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
            out = self.net.extract_features(fb.unsqueeze(1))
        if isinstance(out, dict):
            out = out["x"] if "x" in out else next(iter(out.values()))
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out[:, 1:].float()                                     # without the CLS token


class StandInAttn(nn.Module):
    # TEST only: attention with one fused qkv projection, like EAT's
    def __init__(self, dim, heads=2):
        super().__init__()
        self.h, self.qkv, self.proj = heads, nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))


class StandInBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.n1, self.attn, self.n2 = nn.LayerNorm(dim), StandInAttn(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))


class EatStandIn(nn.Module):
    # TEST only: a small ViT with EAT's interface: extract_features([B, 1, T, 128]) → CLS + (T/16 × 8) patch tokens
    def __init__(self, dim, layers):
        super().__init__()
        self.patch, self.cls = nn.Conv2d(1, dim, 16, 16), nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, 64 * 8, dim) * 0.02)
        self.blocks, self.norm = nn.ModuleList(StandInBlock(dim) for _ in range(layers)), nn.LayerNorm(dim)

    def extract_features(self, mel):
        x = self.patch(mel).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x + self.pos[:, :x.shape[1]]], 1)
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


def onedrive_direct(share_url):
    # OneDrive share link → direct download (the documented 'shares' form of the OneDrive API), as notebook 1
    b64 = base64.b64encode(share_url.encode()).decode().rstrip("=").replace("/", "_").replace("+", "-")
    return f"https://api.onedrive.com/v1.0/shares/u!{b64}/root/content"


def attach_beats_lora(m):
    lora = []
    for layer in m.encoder.layers:
        att = layer.self_attn
        for nm in ("q_proj", "v_proj"):
            w = LoRASwitch(getattr(att, nm), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(att, nm, w); lora.append(w)
    return lora


WN = ((".parametrizations.weight.original0", ".weight_g"), (".parametrizations.weight.original1", ".weight_v"))


def tensors_of(ck):
    # the weights of a loaded checkpoint (a bare state dict, or under "model" or "state_dict") and its configuration, if it has one
    if isinstance(ck, dict):
        for k in ("model", "state_dict"):
            if isinstance(ck.get(k), dict):
                return ck[k], ck.get("cfg")
        if ck and all(torch.is_tensor(v) for v in ck.values()):
            return ck, None
    raise RuntimeError(f"no weights in the file (it holds a {type(ck).__name__}" + (f" with keys {list(ck)[:5]})" if isinstance(ck, dict) else ")"))


def wn_names(sd, own_keys):
    # torch names weight norm's two tensors in two ways (weight_g / weight_v, and parametrizations); the file's keys take the model's
    old = any(k.endswith(".weight_g") for k in own_keys)
    out = {}
    for k, v in sd.items():
        for a, b in WN:
            k = k.replace(a, b) if old else k.replace(b, a)
        out[k] = v
    return out


def match_weights(net, sd):
    # Amendment 1: the file's keys on the model's own, as they are or with one leading prefix removed or added (whichever matches
    # the most keys). Every trainable parameter must come from the file; fixed tensors (buffers, and parameters without gradient)
    # may come from the model's own construction; keys of the file that the model does not have are not used. Both are listed.
    own = net.state_dict(); keys = set(own)
    trainable = {n for n, q in net.named_parameters() if q.requires_grad}
    sd = wn_names(sd, keys)
    cands = {("as is", "")}
    for k in sd:
        for o in keys:
            if k.endswith("." + o):
                cands.add(("removed", k[:len(k) - len(o)]))
            elif o.endswith("." + k):
                cands.add(("added", o[:len(o) - len(k)]))

    def mapping(how, p):
        return {x[len(p):]: x for x in sd if x.startswith(p)} if how == "removed" else {p + x: x for x in sd}

    how, p = max(sorted(cands), key=lambda c: (len(set(mapping(*c)) & keys), c[0] == "as is"))
    m = {k: x for k, x in mapping(how, p).items() if k in keys}
    other = [k for k in m if tuple(sd[m[k]].shape) != tuple(own[k].shape)]
    missing = sorted(keys - set(m)); lost = [k for k in missing if k in trainable]
    unused = sorted(set(sd) - set(m.values()))
    if other or lost:
        raise RuntimeError(f"the file's weights do not fit the model: {len(lost)} trainable parameters are not in the file (e.g. {lost[:3]}), "
                           f"{len(other)} have another shape (e.g. {other[:3]})")
    rep = {"keys_loaded": len(m), "key_prefix": f"'{p}' {how}" if p else "none", "fixed_tensors_from_construction": missing,
           "file_keys_not_used": len(unused), "file_keys_not_used_e_g": unused[:5]}
    return {k: sd[x] for k, x in m.items()}, rep


def load_matched(net, sd):
    state, rep = match_weights(net, sd)
    res = net.load_state_dict(state, strict=False)
    assert not res.unexpected_keys and sorted(res.missing_keys) == rep["fixed_tensors_from_construction"], res
    return rep


def beats_iter3_file():
    # a BEATs iter3 file already in Drive is used as it is; otherwise the authors' link, the Azure copy, then PretrainedSED's copy
    spec = ENCODERS["BEATs-iter3"]; d = WORK if TEST else CKPT_DIR
    own, sed = d / spec["file"], d / spec["sed_file"]
    for f in (own, sed):
        if f.exists() and f.stat().st_size < 10_000_000:
            f.unlink()                                                # an error page saved by a failed attempt, not the model
    if not (own.exists() or sed.exists()):
        for label, url, dest in (("the authors' link (OneDrive)", spec["share"] and onedrive_direct(spec["share"]), own),
                                 ("Microsoft's Azure copy", spec["azure"], own), ("PretrainedSED's release v0.0.1", spec["sed"], sed)):
            if not url:
                continue
            try:
                download(url, dest)
            except Exception as e:
                print(f"BEATs iter3 · {label}: failed ({e.__class__.__name__})")
            if dest.exists() and dest.stat().st_size < 10_000_000:
                dest.unlink()
            if dest.exists():
                print(f"BEATs iter3 · {label}: downloaded", flush=True)
                break
    for f, label in ((own, "the authors' file"), (sed, "PretrainedSED's copy (release v0.0.1); provenance caveat in Amendment 1")):
        if f.exists():
            return f, label
    raise RuntimeError("no download of BEATs iter3 worked (the authors' link, the Azure copy, PretrainedSED's copy)")


def load_beats_iter3():
    f, label = beats_iter3_file()
    ck = torch.load(f, map_location="cpu", weights_only=True)          # tensors and plain values only: no code in the file runs
    sd, cfg_f = tensors_of(ck)
    cfg_ = cfg_f if cfg_f is not None else {**chk["cfg"], "finetuned_model": False}   # weights only: our iter3+ architecture, no classifier
    m = BEATs(BEATsConfig(cfg_))
    rep = load_matched(m, sd)
    msd, ref = m.state_dict(), chk["model"]
    com = [k for k in msd if k in ref and tuple(ref[k].shape) == tuple(msd[k].shape)]
    dist = math.sqrt(sum(float((msd[k].double() - ref[k].double()).pow(2).sum()) for k in com)
                     / max(sum(float(ref[k].double().pow(2).sum()) for k in com), 1e-30))
    m.eval().to(DEVICE)
    for q in m.parameters():
        q.requires_grad_(False)
    info = {"checkpoint_sha256": sha256(f), "file": f.name, "source": label,
            "configuration": "the file's own" if cfg_f is not None else "our iter3+ checkpoint's, classifier off (the file holds weights only)",
            "fine_tuned_on_audioset": bool(getattr(BEATsConfig(cfg_), "finetuned_model", False)),
            "relative_distance_to_our_iter3plus": round(dist, 4), "tensors_compared": len(com), **rep}
    print(f"BEATs iter3 · {label} · SHA-256 {info['checkpoint_sha256'][:16]}… · relative distance from our iter3+ weights {dist:.4f}")
    return BeatsEnc("BEATs-iter3", m, attach_beats_lora(m), info)


def eat_snapshot(repo):
    # the repository's files at one commit: configuration, code and weights (TEST: a local folder)
    if os.path.isdir(repo):
        return repo, "a local folder (TEST)"
    from huggingface_hub import HfApi, snapshot_download
    names = HfApi().list_repo_files(repo)
    pats = ["*.json", "*.py", "*.txt"] + (["*.safetensors"] if any(n.endswith(".safetensors") for n in names) else ["*.bin"])
    path = snapshot_download(repo, allow_patterns=pats)
    return path, Path(path).name                                      # the snapshot folder is named by its commit


def read_weights(path):
    # every weights file of the snapshot: safetensors, else pytorch_model*.bin read with weights_only=True
    files, sd = sorted(Path(path).glob("*.safetensors")), {}
    if files:
        from safetensors.torch import load_file
        for f in files:
            sd.update(load_file(str(f)))
    else:
        files = sorted(Path(path).glob("pytorch_model*.bin"))
        for f in files:
            sd.update(torch.load(f, map_location="cpu", weights_only=True))
    if not sd:
        raise RuntimeError("no weights file (safetensors or pytorch_model.bin) in the repository")
    return sd, [f.name for f in files]


def load_remote_model(path):
    # from_pretrained first; if it fails (transformers 5 with remote code that does not call post_init), the model is built from
    # the repository's own configuration and code, and its weights file is loaded into it (Amendment 1). A package that its
    # code needs is installed, then the load is tried again.
    from transformers import AutoConfig, AutoModel
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    for _ in range(4):
        try:
            try:
                return AutoModel.from_pretrained(path, trust_remote_code=True), {"loaded_by": "from_pretrained"}
            except ImportError:
                raise
            except Exception as e:
                first = repr(e)[:300]
            print(f"  from_pretrained failed ({first[:140]}); building the model from its own configuration and code", flush=True)
            conf = AutoConfig.from_pretrained(path, trust_remote_code=True)
            net = get_class_from_dynamic_module(conf.auto_map["AutoModel"], path)(conf)
            sd, files = read_weights(path)
            rep = load_matched(net, sd)
            return net, {"loaded_by": "built from its own configuration and code; weights loaded directly", "from_pretrained_error": first,
                         "weights_files": files, **rep}
        except ImportError as e:
            if isinstance(e, ModuleNotFoundError) and e.name:
                need = [e.name.split(".")[0]]
            else:
                need = " ".join(re.findall(r"pip install ([^`\n]+)", str(e))[:1]).split()
            if not need:
                raise
            print(f"  its code needs {need}; installing", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
            importlib.invalidate_caches()
    raise RuntimeError("did not load")


def load_eat(key):
    spec = ENCODERS[key]
    if TEST and not os.path.isdir(str(spec.get("repo"))):
        torch.manual_seed(ENC_ORDER.index(key)); net, info = EatStandIn(spec["dim"], spec["layers"]), {"commit": "TEST stand-in"}
    else:
        try:
            import transformers
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
            import transformers
        path, commit = eat_snapshot(spec["repo"])
        net, info = load_remote_model(path)
        info.update(commit=commit, transformers=transformers.__version__)
    net = net.float().to(DEVICE).eval()
    for q in net.parameters():
        q.requires_grad_(False)
    info = {"repo": spec.get("repo"), **info, "parameters_millions": round(sum(q.numel() for q in net.parameters()) / 1e6, 1)}
    fused = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] == "qkv"]
    split = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in ("q_proj", "v_proj", "query", "value")]
    lora = []
    if len(fused) == spec["layers"]:
        for n, m in fused:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRAQKV(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(fused)} fused query-key-value projections (query and value parts)"
    elif len(split) == 2 * spec["layers"]:
        for n, m in split:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRASwitch(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(split)} separate query and value projections"
    else:
        info["lora_problem"] = f"found {len(fused)} fused qkv and {len(split)} separate query/value projections; {spec['layers']} layers expected"
    return EatEnc(key, net, lora, info)


def lora_params_of(enc):
    return [p for w in enc.lora for p in (w.A, w.B)]


def smoke(enc, timed=True):
    # tokens: shape, finiteness; EAT: other lengths; LoRA: unchanged at zero, a gradient reaches it, the switch works; a timed step
    rep = {}
    x10 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (2, 10 * SR)).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        h = enc.tokens(x10, amp=False)
    tp = EAT_FRAMES // 16 if isinstance(enc, EatEnc) else kaldi_frames(10 * SR) // 16
    rep["tokens_of_10_s"] = list(h.shape)
    if not (h.shape[1] == tp * 8 and h.shape[2] == enc.dim and bool(torch.isfinite(h).all())):
        raise RuntimeError(f"{enc.key}: tokens {tuple(h.shape)}, expected (2, {tp * 8}, {enc.dim}) and finite")
    if isinstance(enc, EatEnc):
        enc.varlen = True
        try:
            with torch.no_grad():
                hc = enc.tokens(x10[:, :CROP], amp=False)
            enc.varlen = bool(hc.shape[1] == 16 * math.ceil(kaldi_frames(CROP) / 16) // 16 * 8 and torch.isfinite(hc).all())
        except Exception as e:
            enc.varlen = False; rep["other_length_error"] = repr(e)[:200]
        rep["accepts_other_lengths"] = enc.varlen
    if not enc.lora:
        enc.ok_lora = False; rep["lora"] = enc.info.get("lora_problem", "no LoRA"); return rep
    try:
        xs = x10[:, :CROP]
        with torch.no_grad():
            LoRASwitch.on = False; z0 = enc.pool(enc.tokens(xs, amp=False), CROP)
            LoRASwitch.on = True; z1 = enc.pool(enc.tokens(xs, amp=False), CROP)
        scale = float(z0.abs().mean()) + 1e-12
        assert float((z1 - z0).abs().max()) / scale < 1e-4, "LoRA at zero changed the output"
        head_t = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
        arc_t = SubCenterArcFace(CFG["emb_dim"], 8, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
        opt_t = torch.optim.AdamW(lora_params_of(enc) + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
        enc.train(); head_t.train(); arc_t.train(); times = []
        for k in range(3):
            xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
            yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            loss = arc_t.loss(arc_t.cosine(head_t(enc.pool(enc.tokens(xb, amp=AMP), CROP))), yb)
            opt_t.zero_grad(set_to_none=True); loss.backward()
            if k == 0:
                assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in enc.lora if w.B.grad is not None) > 0, "no gradient reaches LoRA"
            opt_t.step()
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        enc.eval()
        with torch.no_grad():
            moved = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = False
            off = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = True
        assert moved > 1e-6 and off < 1e-4, f"the LoRA switch does not work (moved {moved:.1e}, off {off:.1e})"
        enc.step_s = min(times[1:]) if timed else None
        rep["lora"] = enc.info.get("lora_on", "LoRA"); rep["step_seconds"] = round(enc.step_s, 4) if enc.step_s else None
        del head_t, arc_t, opt_t
    except Exception as e:
        enc.ok_lora = False; rep["lora"] = f"failed: {repr(e)[:300]}"
    finally:
        LoRASwitch.on = True; enc.reset_lora(); enc.eval()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rep


def gate_gb(enc):
    # GB: extra energy in the top (bottom) 16 mel bins over the middle third, against the same input without it; the change of
    # each token, averaged per frequency row under each candidate layout
    with torch.no_grad():
        base = enc.wave2fb(torch.from_numpy(np.random.default_rng(5).normal(0, 0.003, (1, 10 * SR)).astype(np.float32)).to(DEVICE))
        a, b = base.shape[1] // 3, 2 * base.shape[1] // 3
        top, bot = base.clone(), base.clone()
        top[:, a:b, 112:128] += 2.0; bot[:, a:b, 0:16] += 2.0
        LoRASwitch.on = False
        try:
            h0, ht, hb = (enc.fb2tok(z, amp=False) for z in (base, top, bot))
        finally:
            LoRASwitch.on = True
    Lt = h0.shape[1]; Tp = Lt // 8
    dt = (ht - h0).norm(dim=-1)[0].cpu().numpy(); db = (hb - h0).norm(dim=-1)[0].cpu().numpy()
    out = {}
    for lay, row in (("frequency fastest", np.arange(Lt) % 8), ("time fastest", np.arange(Lt) // Tp)):
        pt = np.array([dt[row == f].mean() for f in range(8)]); pb = np.array([db[row == f].mean() for f in range(8)])
        it, ib = int(pt.argmax()), int(pb.argmax())
        rt = float(pt[it] / max(np.delete(pt, it).mean(), 1e-12)); rb = float(pb[ib] / max(np.delete(pb, ib).mean(), 1e-12))
        out[lay] = {"top_row": it, "bottom_row": ib, "ratio_top": round(rt, 2), "ratio_bottom": round(rb, 2),
                    "confirmed": bool({it, ib} == {0, 7} and min(rt, rb) >= GB_RATIO)}   # the two ends of the spectrum, one each
    if out["time fastest"]["confirmed"] and not out["frequency fastest"]["confirmed"]:
        enc.layout = "time fastest"
    out["used"] = enc.layout
    return out


ENC, FAILED, GBR = {}, {}, {}
ENC["BEATs"] = BeatsEnc("BEATs", model, LORA, {"checkpoint_sha256": CKPT_SHA})        # cell 9's model and LoRA blocks
ENC["BEATs"].step_s = STEP_S
with torch.no_grad():
    _gap = float((ENC["BEATs"].tokens(x4, amp=False) - tokens(x4, amp=False)).abs().max())
    _pool = float((ENC["BEATs"].pool(tokens(x4, amp=False), x4.shape[1]) - tokens(x4, amp=False).mean(1)).abs().max())
print(f"BEATs through the encoder wrapper against cell 9's forward pass: {_gap:.1e} (pooling {_pool:.1e})")
assert _gap < 1e-6 and _pool < 1e-5, "stop: the encoder wrapper does not reproduce cell 9"
for key in NEW_ENCODERS:
    t0 = time.time()
    try:
        enc = load_beats_iter3() if key == "BEATs-iter3" else load_eat(key)
        enc.info.update(smoke(enc))
        ENC[key] = enc
        print(f"{key}: ready · {json.dumps(enc.info)} · {time.time() - t0:.0f} s", flush=True)
    except Exception as e:
        FAILED[key] = repr(e)[:500]
        print(f"{key}: DROPPED, our loader failed ({FAILED[key]})", flush=True)
    gc.collect()
for key, enc in ENC.items():
    GBR[key] = gate_gb(enc)
    print(f"GB · {key}: {json.dumps(GBR[key])}")


# G0(h): the BEATs iter3 file is the one notebook 22b used
G0H = {"sha256": ENC["BEATs-iter3"].info.get("checkpoint_sha256") if "BEATs-iter3" in ENC else None, "expected": B3_SHA_N22B}
print("G0(h) · the BEATs iter3 file is notebook 22b's:", G0H["sha256"] == B3_SHA_N22B)
if not TEST and G0H["sha256"] != B3_SHA_N22B:
    raise SystemExit("stop: G0(h) failed, the BEATs iter3 file is not the one notebook 22b used (or it did not load)")

In [ ]:
# 8 · ECHO, frozen (notebook 23's cell, unchanged; notebook 20's cell: notebook 15's loader, as on the model card); gate G0(g), its weights
INFO = {}
def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()




@torch.no_grad()
def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class EchoStandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like ECHO's output. Never used on Colab.
    def __init__(self, seed=2, bands=7, dim=16):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def extract_features_from_audio(self, x, sample_rate):
        S = torch.stft(x[0].float(), n_fft=400, hop_length=160, window=torch.hann_window(400, device=x.device), return_complex=True).abs().log1p().T
        e = torch.stack([b.mean(-1) for b in S.chunk(self.bands, dim=-1)], -1)
        q = torch.stack([e[i * e.shape[0] // 8:(i + 1) * e.shape[0] // 8].mean(0) for i in range(8)], 0)
        return torch.einsum("qk,kqd->kd", q, self.P.to(x.device)).flatten(), None


if TEST:
    ECHO = EchoStandIn().to(DEVICE).eval(); INFO["ECHO"] = {"repo": "stand-in (TEST)", "weights_sha256": "", "loader": "stand-in"}
else:
    ECHO = load_echo()
    if INFO["ECHO"]["weights_sha256"] != ECHO_SHA_N15:
        print("NOTE: the ECHO weights differ from notebook 15's (gate G0(g))")

G0G = {"weights_sha256": INFO["ECHO"].get("weights_sha256"), "expected": ECHO_SHA_N15, "revision": INFO["ECHO"].get("revision")}
print("ECHO:", json.dumps(INFO["ECHO"]))
if not TEST and G0G["weights_sha256"] != ECHO_SHA_N15:
    raise SystemExit("stop: G0(g) failed, the ECHO weights are not notebook 15's")


@torch.no_grad()
def echo_stream(mm, rows=None, chunk=64):
    # ECHO on whole clips of a stream, a few at a time → (n, 7 × 384) float32
    rows = np.arange(mm.shape[0]) if rows is None else rows
    return np.concatenate([emb_echo(ECHO, [f32(mm[r]) for r in rows[i:i + chunk]]) for i in range(0, len(rows), chunk)])

# gate G0(c) of this notebook's rules: the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's (SHA-256).
# The two checkpoints and ECHO were already checked by the cells above; the code files are checked here
G0C = {"code_files": {f: bool(code_sha[f] == h) for f, h in CODE_SHA256.items()}, "beats_iter3plus": bool(CKPT_SHA == CKPT_SHA_N1),
       "beats_iter3": bool(G0H["sha256"] == B3_SHA_N22B), "echo_weights": bool(G0G["weights_sha256"] == ECHO_SHA_N15)}
G0C["ok"] = bool(all(G0C["code_files"].values()) and G0C["beats_iter3plus"] and G0C["beats_iter3"] and G0C["echo_weights"])
print("G0(c) · the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's:", G0C["ok"], G0C)
if not TEST and not G0C["ok"]:
    raise SystemExit("stop: G0(c) failed, a code file or a model file is not the one notebook 23 used")

In [ ]:
# 9 · DCASE 2025's own files of the fifteen types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store per
# type on the local disk; gates G0(a) and G0(b). The readers are notebook 23's (through notebook 25). The loader is notebook 28's: a
# development type has one zip, an evaluation type two; the supplemental clips are not read.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


WAV_FORMATS = Counter()                                               # the sample formats of the WAV files read in this session


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE; PCM of 16, 24 or 32 bits, or 32-bit float) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert (tag, bits) in ((1, 16), (1, 24), (1, 32), (3, 32)), f"expected PCM of 16, 24 or 32 bits or 32-bit float, got format {tag}, {bits}-bit"
    WAV_FORMATS[f"{'float' if tag == 3 else 'PCM'} {bits}-bit"] += 1
    w = bits // 8; nf = len(raw) // (w * ch); raw = raw[:nf * w * ch]
    if (tag, bits) == (1, 16):
        return np.frombuffer(raw, dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr
    if (tag, bits) == (1, 24):                                        # three bytes per sample, little-endian, signed: a 24-bit value is exact in float32
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        v = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16); v = np.where(v >= 1 << 23, v - (1 << 24), v)
        return (v.astype(np.float32) / 8388608.0).reshape(-1, ch), sr
    if (tag, bits) == (1, 32):
        return (np.frombuffer(raw, dtype="<i4").astype(np.float64) / 2147483648.0).astype(np.float32).reshape(-1, ch), sr
    return np.frombuffer(raw, dtype="<f4").reshape(-1, ch).astype(np.float32), sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # exact for a 16-bit WAV file; a clip of more bits is rounded to 16 (to the nearest, ties to even) and clipped to the int16 range
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores30"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2025 clip of type t in one int16 store of NS_RULE samples (10 s): the first 10 s of a longer clip, as the view's reference
    # code cuts it, or a shorter clip repeated from its start; a development type's train and test clips, or an evaluation type's training clips and its test clips. The supplemental clips are not read
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {k_: zipfile.ZipFile(get_file(z_[t])) for k_, z_ in (("dev", DC_DEV), ("evaltrain", XZIPS), ("evaltest", DC_EVAL)) if t in z_}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            if folder not in ("train", "test"):                       # the supplemental clips (clean machine sound or noise only) are not used
                continue
            part = folder
            # the name of a training clip and of a development test clip carries its domain and its kind; an evaluation test clip's carries neither
            said = re.match(r"section_\d+_(source|target)_(train|test)_(normal|anomaly)_", base)
            label = (1 if (said and said.group(3) == "anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            dom = said.group(1) if said else ""
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(section_\d+)", base).group(1), label, base, dom))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The sections of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and "_train_normal_" not in r[6] for r in rows), "the additional training clips are normal"
    ns, lengths, _f0 = NS_RULE, Counter(), Counter(WAV_FORMATS)
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= NS_MIN):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {NS_MIN / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        w_ = to_int16(x[:ns, 0])
        mm[i] = w_ if len(w_) == ns else np.tile(w_, -(-ns // len(w_)))[:ns]   # a shorter clip is repeated from its start until it fills the store
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "domain": [r[7] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "formats": {k: int(v - _f0[k]) for k, v in sorted(WAV_FORMATS.items()) if v - _f0[k]},
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in [z_[t] for z_ in (DC_DEV, XZIPS, DC_EVAL) if t in z_]:
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}, "domains": {}}
for t in TYPES:
    dcase_load(t)
    _m = DMETA[t]; _mid, _dom = np.array(_m["mid"]), np.array(_m["domain"])
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["domains"][t] = {f"{s_}|{p_}": {q: {d_: int((_dom[dc_rows(t, s_, p_, q)] == d_).sum()) for d_ in ("source", "target")}
                                          for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))} for s_, p_ in (("dev", "train"), ("dev", "test"), ("eval", "train"))}
    DCASE["lengths"][t] = _m["lengths"]; DCASE.setdefault("formats", {})[t] = _m.get("formats", "not recorded (a store of an earlier session)")
    print(f"DCASE 2025 · {t}: {len(_m['name'])} clips · training clips per section (source, target): "
          f"{({q: (v['source'], v['target']) for k_ in ('dev|train', 'eval|train') for q, v in DCASE['domains'][t][k_].items()})} · test clips per section: "
          f"{({**DCASE['counts'][t]['dev|test'], **DCASE['counts'][t]['eval|test']})} · samples per clip before the cut (set|part: samples: clips): {DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {k_: list(z_[t][1:]) for k_, z_ in (("dev", DC_DEV), ("additional", XZIPS), ("evaluation", DC_EVAL)) if t in z_} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print(f"G0(a) · the {sum(len(v) for v in _want.values())} zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def g0b(t):
    # the files are what the rules assume (16 kHz, one channel and 10 s were checked per file by the loader)
    m = DMETA[t]; nm, st, pt, lb, mid, dom = (np.array(m[k]) for k in ("name", "set", "part", "label", "mid", "domain"))
    c = DCASE["counts"][t]; tr, ev_test = pt == "train", (st == "eval") & (pt == "test")
    assert (DEV_IDS[t] == [SECTION]) != (EVAL_IDS[t] == [SECTION])                # a type is a development type or an evaluation type
    first_target = {}                                                 # per training section: how many source-domain clips come before its first target-domain clip by name
    for s_, ids in (("dev", DEV_IDS[t]), ("eval", EVAL_IDS[t])):
        for q in ids:
            rows = dc_rows(t, s_, "train", q); d_ = dom[rows[np.argsort(nm[rows], kind="stable")]]
            first_target[q] = int(np.argmax(d_ == "target")) if (d_ == "target").any() else -1
    both = all(int(((st == "dev") & (pt == "test") & (mid == q) & (lb == k_) & (dom == d_)).sum()) > 0 for q in DEV_IDS[t] for k_ in (0, 1) for d_ in ("source", "target"))
    _num = sorted(int(n_[-8:-4]) for n_ in nm[ev_test] if re.fullmatch(r"section_\d\d_\d{4}\.wav", n_))   # 2025's names: section_00_NNNN.wav, from 0000 or from 0001
    numbered = (not EVAL_IDS[t]) or (len(_num) == int(ev_test.sum()) > 0 and _num == list(range(_num[0], _num[0] + len(_num))))
    out = {"sections": bool(sorted(c["dev|train"]) == sorted(c["dev|test"]) == DEV_IDS[t] and sorted(c["eval|train"]) == sorted(c["eval|test"]) == EVAL_IDS[t]),
           "training_clips_are_normal": bool(all("_train_normal_" in n_ for n_ in nm[tr]) and (lb[tr] == 0).all()),
           "training_clips_carry_their_domain": bool(np.isin(dom[tr], ["source", "target"]).all()),
           "source_clips_come_first_by_name": bool(all(v >= MIN_SOURCE_FIRST for v in first_target.values())),
           "every_training_section_has_target_clips": bool(all(v >= 0 for v in first_target.values())),
           "development_sections_have_both_kinds_in_both_domains": bool(both),
           "development_test_names_carry_label_and_domain": bool(all(("_test_normal_" in n_ or "_test_anomaly_" in n_) and ("_source_" in n_ or "_target_" in n_)
                                                                     for n_ in nm[(st == "dev") & (pt == "test")])),
           "evaluation_test_names_carry_no_label_or_domain": bool((not EVAL_IDS[t]) or (ev_test.any() and (dom[ev_test] == "").all()
                                                                  and not any(w in n_ for n_ in nm[ev_test] for w in ("normal", "anomaly", "source", "target")))),
           "evaluation_test_clips_are_numbered_consecutively": bool(numbered),
           "no_evaluation_label_in_the_store": bool((lb[ev_test] == -1).all()),
           "names_are_unique_in_each_folder": bool(all(len(set(nm[(st == s_) & (pt == p_)].tolist())) == int(((st == s_) & (pt == p_)).sum())
                                                       for s_ in ("dev", "eval") for p_ in ("train", "test"))),
           "clips": int(len(nm)), "skipped_archive_entries": int(len(m["skipped_archive_entries"])), "source_clips_before_the_first_target_clip": first_target,
           "shortest_clip_samples": int(min(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "longest_clip_samples": int(max(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "clips_repeated_to_fill_the_store": int(sum(c_ for d_ in m["lengths"].values() for k_, c_ in d_.items() if int(k_) < NS))}
    out["ok"] = bool(all(v for k, v in out.items() if isinstance(v, bool)) and out["shortest_clip_samples"] >= NS_MIN)
    return out


G0["b"] = {t: g0b(t) for t in TYPES}
print(f"G0(b) · the sections, the clip kinds, the domains and the names are the rules'; every clip is 16 kHz, one channel, at least {NS_MIN / SR:g} s:",
      all(v["ok"] for v in G0["b"].values()), {t: {k: v for k, v in d.items() if v is False} or "all checks true" for t, d in G0["b"].items()})
if not all(v["ok"] for v in G0["b"].values()):
    raise SystemExit(f"stop: G0(b) failed, the files are not what the rules assume: {G0['b']}")
N_TRAIN_ALL, N_TEST_ALL = (int(sum(sum(DCASE["counts"][t][f"{s_}|{p_}"].values()) for t in TYPES for s_ in ("dev", "eval"))) for p_ in ("train", "test"))
PREP_S = round(time.time() - t_data, 1)
print(f"data ready · {N_TRAIN_ALL} training clips and {N_TEST_ALL} test clips · {NS} samples per clip are kept ({NS / SR:g} s) · {PREP_S / 60:.1f} min")
print("sample formats of the WAV files, per type:", DCASE["formats"])
print("clips shorter than that, repeated from their start to fill it:", {t: G0["b"][t]["clips_repeated_to_fill_the_store"] for t in TYPES if G0["b"][t]["clips_repeated_to_fill_the_store"]} or "none")

In [ ]:
# 10 · Notebook 35's own settings (fixed in its rules before it was built): where notebook 30's trainings are read from, the eleven
# poolings of the last layer's tokens, and the pass that pools every clip
RULES35 = "results/mimii_beats/RULES_pooling_first_shot_2025_written_before_running.md"
SRC30 = (MY / "P04_FirstShot") if IN_COLAB else Path(os.environ.get("P04_SRC30", "P04_FirstShot"))   # notebook 30's folder: read, never written
SRC30_AT = SRC30 / "attribute_classes"
ENCT = {"BEATs": "B", "BEATs-iter3": "B3", "EAT-base": "EB", "EAT-large": "EL"}
TRK = {"ARR": "ARR", "MIC+": "MICp", "MIC": "MIC", "DC": "DC"}
GEM_P, RDP_G, RG_G, RG_P = [3, 6, 10], [4, 10, 16, 19], [9, 16], 3
POOLS = ["mean", "max"] + [f"gem{p}" for p in GEM_P] + [f"rdp{g}" for g in RDP_G] + [f"rdpgem{g}" for g in RG_G]
ALPHAS = np.round(np.arange(0.0, 1.5001, 0.05), 2)                   # scorer Ev: the grid of the variance minimisation
N_MIX, N_CROSS = 200, 20                                              # pseudo-outliers per type: feature mixes; training clips of each other type
SECTION_FILE = "section_00"


def n22_tag(ek, track, inp, flags, seed, fold):
    return f"{ENCT[ek]}_{TRK[track]}{'n' if inp == 'N2N-MWF' else ''}_{'-'.join(flags) or 'R0'}_s{seed}_f{fold}"


def load_lora(enc, lst):
    assert len(lst) == len(enc.lora), (len(lst), len(enc.lora))
    with torch.no_grad():
        for w, s in zip(enc.lora, lst):
            w.A.copy_(s["A"].to(DEVICE)); w.B.copy_(s["B"].to(DEVICE))


AT_TAGS = {(ek, s): n22_tag(ek, "DC", "CH0", ["XC"], s, 0) for ek, _, seeds in FROZEN for s in seeds}
_need = ([SRC30_AT / f"ft_{g}.pt" for g in AT_TAGS.values()] + [SRC30_AT / f"emb_{g}.npz" for g in AT_TAGS.values()]
         + [SRC30 / "dcase" / f"echo_{t}.npy" for t in TYPES]
         + [SRC30 / "development_scores" / f"scores_{t}_{SECTION_FILE}.csv" for t in TYPES_DEV]
         + [SRC30 / "dcase2025_evaluation" / "frozen_AT" / f"anomaly_score_{t}_{SECTION_FILE}_test.csv" for t in TYPES_EVAL])
_missing = [str(f) for f in _need if not f.exists()]
if _missing:
    raise SystemExit(f"stop: notebook 30's files are not all in {SRC30} ({len(_missing)} missing, for example {_missing[0]}). "
                     "This notebook reads notebook 30's trainings from Drive › P04_FirstShot.")
print(f"notebook 30's folder: {SRC30} · {len(AT_TAGS)} trainings with attribute classes found · poolings: {POOLS}")


def pool_all(h):
    # h (B, L, D) float32: every time-frequency token of a clip → one vector per pooling (arXiv 2603.04605's formulas)
    out, mu = {}, h.mean(1)
    out["mean"], out["max"] = mu, h.amax(1)
    hp = h.clamp_min(0.0); top = hp.amax(1, keepdim=True).clamp_min(1e-12); hn = hp / top     # scaled by the largest value: no overflow for p = 10
    for p in GEM_P:
        out[f"gem{p}"] = hn.pow(p).mean(1).pow(1.0 / p) * top[:, 0]
    d = (h - mu[:, None]).norm(dim=-1); dh = d / d.amax(1, keepdim=True).clamp_min(1e-12)        # each token's deviation from the mean, 0 to 1
    for g in RDP_G:
        w = (1.0 + dh) ** g; w = w / w.sum(1, keepdim=True)
        out[f"rdp{g}"] = (w.unsqueeze(-1) * h).sum(1)
    for g in RG_G:
        w = (1.0 + dh) ** g; w = w / w.sum(1, keepdim=True)
        out[f"rdpgem{g}"] = (w.unsqueeze(-1) * hn.pow(RG_P)).sum(1).pow(1.0 / RG_P) * top[:, 0]
    return out


# the poolings against a slow reference written directly from the formulas
_h = torch.from_numpy(np.random.default_rng(35).normal(0.2, 1.0, (3, 40, 12)).astype(np.float32)); _o = pool_all(_h); _x = _h.double().numpy()
_d = np.linalg.norm(_x - _x.mean(1, keepdims=True), axis=-1); _dh = _d / _d.max(1, keepdims=True); _xp = np.maximum(_x, 0.0)
_ref = {"mean": _x.mean(1), "max": _x.max(1), **{f"gem{p}": (_xp ** p).mean(1) ** (1.0 / p) for p in GEM_P}}
for g in RDP_G:
    _w = (1 + _dh) ** g; _w = _w / _w.sum(1, keepdims=True); _ref[f"rdp{g}"] = (_w[..., None] * _x).sum(1)
for g in RG_G:
    _w = (1 + _dh) ** g; _w = _w / _w.sum(1, keepdims=True); _ref[f"rdpgem{g}"] = ((_w[..., None] * _xp ** RG_P).sum(1)) ** (1.0 / RG_P)
_gap = max(float(np.abs(_o[p].double().numpy() - _ref[p]).max()) for p in POOLS)
print(f"the {len(POOLS)} poolings against their formulas: largest difference {_gap:.1e}")
assert sorted(_o) == sorted(POOLS) and _gap < 1e-4, "stop: a pooling is not its formula"


@torch.no_grad()
def pooled_pass(enc, mm, head=None):
    # whole clips through the encoder as it stands (LoRA as switched): per clip and pooling the pooled tokens P, and head(P) when a head is given
    n, bs = mm.shape[1], CFG["embed_batch"]; enc.eval()
    P = {p: np.zeros((mm.shape[0], enc.dim), np.float16) for p in POOLS}
    Fh = {p: np.zeros((mm.shape[0], CFG["emb_dim"]), np.float16) for p in POOLS} if head is not None else None
    for i in range(0, mm.shape[0], bs):
        x = torch.from_numpy(f32(mm[i:i + bs])).to(DEVICE); m_ = len(x)
        g = enc.grid(enc.tokens(x, amp=AMP), n)
        for p, v in pool_all(g.reshape(m_, -1, g.shape[-1]).float()).items():
            P[p][i:i + m_] = v.cpu().numpy()
            if head is not None:
                Fh[p][i:i + m_] = head(v).float().cpu().numpy()
    return P, Fh

In [ ]:
# 11 · The passes (the slow part; each finished pass is saved in Drive › P04_FirstShot35 › pooled and is not repeated): the two frozen
# encoders with LoRA off, then notebook 30's six trainings with attribute classes with their own LoRA and head. Gate G0(q)
POOL_DIR = OUT / "pooled"; POOL_DIR.mkdir(exist_ok=True)
PASS_S, t_pass = {}, time.time()


def save_npz(path, arrays):
    tmp = path.with_name(path.stem + ".part.npz"); np.savez(tmp, **arrays); os.replace(tmp, path)


def load_state(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)   # our own training file from notebook 30


for ek, code, _ in FROZEN:                                            # frozen: the encoder as downloaded
    f = POOL_DIR / f"frozen_{code}.npz"
    if not f.exists():
        t0 = time.time(); LoRASwitch.on = False
        try:
            out = {}
            for t in TYPES:
                P, _ = pooled_pass(ENC[ek], DST[t])
                out.update({f"P__{p}__{t}": P[p] for p in POOLS})
        finally:
            LoRASwitch.on = True
        save_npz(f, out); PASS_S[f.stem] = round(time.time() - t0, 1)
        print(f"{f.name}: saved · {PASS_S[f.stem] / 60:.1f} min · {(time.time() - T_START) / 60:.0f} min in this session", flush=True)

G0Q = {}
for (ek, s), tag in AT_TAGS.items():                                  # the expert with attribute classes: notebook 30's LoRA and head
    f = POOL_DIR / f"at_{tag}.npz"
    if not f.exists():
        t0 = time.time(); st = load_state(SRC30_AT / f"ft_{tag}.pt")
        load_lora(ENC[ek], st["lora"])
        head = Head(ENC[ek].dim, CFG["emb_dim"]).to(DEVICE); head.load_state_dict(st["head"]); head.eval(); LoRASwitch.on = True
        out = {}
        for t in TYPES:
            P, Fh = pooled_pass(ENC[ek], DST[t], head)
            out.update({f"P__{p}__{t}": P[p] for p in POOLS}); out.update({f"F__{p}__{t}": Fh[p] for p in POOLS})
        save_npz(f, out); ENC[ek].reset_lora(); del head, st
        PASS_S[f.stem] = round(time.time() - t0, 1)
        print(f"{f.name}: saved · {PASS_S[f.stem] / 60:.1f} min · {(time.time() - T_START) / 60:.0f} min in this session", flush=True)
    Z, S30 = np.load(f, allow_pickle=False), np.load(SRC30_AT / f"emb_{tag}.npz", allow_pickle=False)
    worst = 1.0
    for t in TYPES:                                                   # G0(q): the mean pooling through the head is notebook 30's saved embedding
        a, b = Z[f"F__mean__{t}"].astype(np.float64), S30[f"F__{t}"].astype(np.float64)
        assert a.shape == b.shape == (len(DMETA[t]["name"]), CFG["emb_dim"]), f"{tag} {t}: {a.shape} against notebook 30's {b.shape}"
        worst = min(worst, float(((a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))).min()))
    G0Q[tag] = round(worst, 6)
G0Q_OK = bool(min(G0Q.values()) >= 0.999)
print("G0(q) · the mean pooling through each head equals notebook 30's saved embeddings (smallest cosine per training):", G0Q, "→", "pass" if G0Q_OK else "FAIL")
if not TEST and not G0Q_OK:
    raise SystemExit("stop: G0(q) failed, the trainings read from Drive do not reproduce notebook 30's embeddings")
T_PASSES = round(time.time() - t_pass, 1)
for _n in ("model", "chk", "ECHO"):
    globals().pop(_n, None)
ENC.clear(); gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(f"passes done · {T_PASSES / 60:.0f} min in this cell")

In [ ]:
# 12 · The score bank. For every candidate and type, with no evaluation label read: the scores of the test clips, the leave-one-out
# scores of the reference clips (the type's training clips), and the scores of two kinds of pseudo-outliers. Then the development table
def tt(a):
    return torch.from_numpy(np.ascontiguousarray(a, dtype=np.float64)).to(DEVICE)


def euclid_bank(R, Xs):
    # Euclidean nearest neighbour in log distance, with local density normalisation (K = 1): min over references y of
    # log D(x, y) − α · log D(y, y₁). E0: α = 0; E1: α = 1; Ev: α by variance minimisation over the references (arXiv 2603.04605)
    Lrr = torch.log(torch.cdist(R, R).clamp_min(LDN_FLOOR)); Lrr.fill_diagonal_(float("inf"))
    d1 = Lrr.min(1).values
    var = [float(torch.var((Lrr - float(a) * d1[None, :]).min(1).values)) for a in ALPHAS]
    a_star = float(ALPHAS[int(np.argmin(var))]); Lx = [torch.log(torch.cdist(X, R).clamp_min(LDN_FLOOR)) for X in Xs]
    return {k: [(Lrr - a * d1[None, :]).min(1).values] + [(L - a * d1[None, :]).min(1).values for L in Lx]
            for k, a in (("E0", 0.0), ("E1", 1.0), ("Ev", a_star))}, a_star


def cos_bank(R, Xs):
    # this project's LDN1 (notebook 30's ldn1): min over references y of (1 − cos(x, y)) / (1 − cos(y, y₁))
    U = F.normalize(R, dim=1); S = U @ U.T; S.fill_diagonal_(-float("inf"))
    d1 = (1.0 - S.max(1).values).clamp_min(LDN_FLOOR)
    return [((1.0 - S) / d1[None, :]).min(1).values] + [((1.0 - F.normalize(X, dim=1) @ U.T) / d1[None, :]).min(1).values for X in Xs]


# notebook 30's ldn1, as it is written there, against cos_bank
_R, _X = np.random.default_rng(11).standard_normal((40, 9)), np.random.default_rng(12).standard_normal((7, 9))
_u = lambda E: E / np.linalg.norm(E, axis=-1, keepdims=True)
_Srr = _u(_R) @ _u(_R).T; np.fill_diagonal(_Srr, -np.inf); _d1 = np.maximum(1.0 - _Srr.max(1), LDN_FLOOR)
assert np.allclose(cos_bank(tt(_R), [tt(_X)])[1].cpu().numpy(), ((1.0 - _u(_X) @ _u(_R).T) / _d1[None, :]).min(1)), "stop: cos_bank is not notebook 30's LDN1"

SET_OF = {t: ("dev" if t in TYPES_DEV else "eval") for t in TYPES}
TR = {t: dc_rows(t, SET_OF[t], "train") for t in TYPES}; TE = {t: dc_rows(t, SET_OF[t], "test") for t in TYPES}
CROSS = {t: TR[t][np.sort(np.random.default_rng([35, 2, i]).choice(len(TR[t]), min(N_CROSS, len(TR[t])), replace=False))] for i, t in enumerate(TYPES)}
MIXAB = {t: np.random.default_rng([35, 1, i]).integers(0, len(TR[t]), (2, N_MIX)) for i, t in enumerate(TYPES)}


def unit32(E):
    E = E.astype(np.float32)
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def four(M, t, ti, bank_fn):
    # M: {type: (clips, d)} one listener's embedding of every clip → for type t the scores of [references leave-one-out, test, mixes, other types]
    R = M[t][TR[t]]; a, b = MIXAB[t]
    mask = np.random.default_rng([35, 3, ti]).random((N_MIX, R.shape[1])) < 0.5
    mix = np.where(mask, R[a], R[b]); cross = np.concatenate([M[u][CROSS[u]] for u in TYPES if u != t])
    return bank_fn(tt(R), [tt(M[t][TE[t]]), tt(mix), tt(cross)])


BANK = {t: {} for t in TYPES}                                         # type → candidate key → [loo, test, mix, cross] as float32 arrays
ALPHA_STAR, t_bank = {}, time.time()
put = lambda t, key, four_: BANK[t].__setitem__(key, [v.float().cpu().numpy() if torch.is_tensor(v) else np.asarray(v, np.float32) for v in four_])

for _, code, _ in FROZEN:                                             # frozen encoders: 11 poolings × 4 scorers
    Z = np.load(POOL_DIR / f"frozen_{code}.npz", allow_pickle=False)
    for p in POOLS:
        M = {t: Z[f"P__{p}__{t}"].astype(np.float32) for t in TYPES}
        for ti, t in enumerate(TYPES):
            eb, a_star = four(M, t, ti, euclid_bank)
            for k in ("E0", "E1", "Ev"):
                put(t, f"FZ|{code}|{p}|{k}", eb[k])
            ALPHA_STAR[f"{code}|{p}|{t}"] = a_star
            put(t, f"FZ|{code}|{p}|C1", four(M, t, ti, cos_bank))
    print(f"frozen {code}: banked · {(time.time() - t_bank) / 60:.1f} min", flush=True)

ECHO_DC = {t: np.load(SRC30 / "dcase" / f"echo_{t}.npy") for t in TYPES}
assert all(len(ECHO_DC[t]) == len(DMETA[t]["name"]) for t in TYPES), "stop: notebook 30's ECHO embeddings do not have this session's number of clips"
for ti, t in enumerate(TYPES):
    put(t, "ECHO|C1", four({u: ECHO_DC[u].astype(np.float32) for u in TYPES}, t, ti, cos_bank))
ZAT = {tag: np.load(POOL_DIR / f"at_{tag}.npz", allow_pickle=False) for tag in AT_TAGS.values()}
for p in POOLS:                                                       # the expert with attribute classes: 11 poolings × 3 forms
    for form in ("F", "Fx", "P"):
        parts = []
        for ek, code, seeds in FROZEN:                                # one listener per encoder: its three seeds side by side (notebook 30's frozen_matrix)
            M = {}
            for t in TYPES:
                E = np.concatenate([unit32(ZAT[AT_TAGS[(ek, s)]][f"{'P' if form == 'P' else 'F'}__{p}__{t}"]) for s in seeds], 1) / math.sqrt(len(seeds))
                M[t] = np.concatenate([E, unit32(ECHO_DC[t])], 1) / math.sqrt(2.0) if form == "F" else E
            parts.append(M)
        for ti, t in enumerate(TYPES):                                # the expert's score is the mean of its two encoders' scores, as in notebook 30
            fours = [four(M, t, ti, cos_bank) for M in parts]
            put(t, f"AT|{form}|{p}", [sum(f4[j] for f4 in fours) / len(fours) for j in range(4)])
    print(f"expert with attribute classes, pooling {p}: banked · {(time.time() - t_bank) / 60:.1f} min", flush=True)
KEYS = list(BANK[TYPES[0]]); assert all(list(BANK[t]) == KEYS for t in TYPES)
print(f"{len(KEYS)} candidates banked for {len(TYPES)} types · {(time.time() - t_bank) / 60:.1f} min")

# G0(r): candidate AT|F|mean is notebook 30's expert with attribute classes (its saved score files)
def ranks(v):
    r = np.empty(len(v)); r[np.argsort(v, kind="stable")] = np.arange(len(v)); return r


G0R = {}
for t in TYPES:
    names = np.array(DMETA[t]["name"])[TE[t]]; mine = dict(zip(names.tolist(), BANK[t]["AT|F|mean"][1].tolist()))
    if t in TYPES_DEV:
        with open(SRC30 / "development_scores" / f"scores_{t}_{SECTION_FILE}.csv", newline="") as fh:
            saved = {r["file"]: float(r["frozen_AT"]) for r in csv.DictReader(fh)}
    else:
        with open(SRC30 / "dcase2025_evaluation" / "frozen_AT" / f"anomaly_score_{t}_{SECTION_FILE}_test.csv", newline="") as fh:
            saved = {r[0]: float(r[1]) for r in csv.reader(fh) if r}
    assert sorted(saved) == sorted(mine), f"{t}: the test clips are not notebook 30's"
    a, b = np.array([mine[k] for k in sorted(mine)]), np.array([saved[k] for k in sorted(mine)])
    G0R[t] = {"spearman": round(float(np.corrcoef(ranks(a), ranks(b))[0, 1]), 6), "largest_relative_difference": round(float(np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-12))), 5)}
G0R_OK = bool(min(v["spearman"] for v in G0R.values()) >= 0.999)
print("G0(r) · AT|F|mean against notebook 30's saved scores, per type:", G0R, "→", "pass" if G0R_OK else "FAIL")
if not TEST and not G0R_OK:
    raise SystemExit("stop: G0(r) failed, the bank does not reproduce notebook 30's expert with attribute classes")


# the development table (seven types with labels in their file names): the official score of every candidate
def three(y, s, dom):
    y, s, dom = np.asarray(y), np.asarray(s, float), np.asarray(dom); an = y == 1
    return [float(roc_auc_score(y[((dom == d) & ~an) | an], s[((dom == d) & ~an) | an])) for d in ("source", "target")] + [float(roc_auc_score(y, s, max_fpr=0.1))]


hmean = lambda v: len(v) / sum(1.0 / max(float(x), sys.float_info.epsilon) for x in v)
DEV = {}
for k in KEYS:
    vals = []
    for t in TYPES_DEV:
        y, dom = np.array(DMETA[t]["label"])[TE[t]], np.array(DMETA[t]["domain"])[TE[t]]
        vals.append(three(y, BANK[t][k][1], dom))
    v = np.array(vals)
    DEV[k] = {"omega": round(hmean(v.ravel().tolist()), 4), "AUC_source": round(hmean(v[:, 0].tolist()), 4), "AUC_target": round(hmean(v[:, 1].tolist()), 4), "pAUC": round(hmean(v[:, 2].tolist()), 4)}
print("\ndevelopment set · official score (source line, target line, pAUC line) · the 20 highest candidates, then the named ones")
for k in sorted(KEYS, key=lambda k: -DEV[k]["omega"])[:20]:
    print(f"  {k:<24s} {DEV[k]['omega']:.4f}  ({DEV[k]['AUC_source']:.4f}, {DEV[k]['AUC_target']:.4f}, {DEV[k]['pAUC']:.4f})")
for k in ("AT|F|mean", "FZ|B3|rdpgem16|Ev", "FZ|B3|mean|Ev", "FZ|B|rdpgem16|Ev", "ECHO|C1"):
    print(f"  named · {k:<16s} {DEV[k]['omega']:.4f}  ({DEV[k]['AUC_source']:.4f}, {DEV[k]['AUC_target']:.4f}, {DEV[k]['pAUC']:.4f})")
print("(notebook 30's record: the expert with attribute classes, development 0.5528; arXiv 2603.04605 reports 64.0 for frozen BEATs with RDP+GeM)")

In [ ]:
# 13 · Save the bank (one file per type), the record of this run, and one zip to download. No evaluation label was read
BANK_DIR = OUT / "bank"; BANK_DIR.mkdir(exist_ok=True)
for t in TYPES:
    names, ref_names = np.array(DMETA[t]["name"])[TE[t]], np.array(DMETA[t]["name"])[TR[t]]
    arr = {"keys": np.array(KEYS), "test_names": names, "reference_names": ref_names,
           "loo": np.stack([BANK[t][k][0] for k in KEYS]), "test": np.stack([BANK[t][k][1] for k in KEYS]),
           "mix": np.stack([BANK[t][k][2] for k in KEYS]), "cross": np.stack([BANK[t][k][3] for k in KEYS])}
    assert all(np.isfinite(arr[k_]).all() for k_ in ("loo", "test", "mix", "cross")), f"{t}: a score is not finite"
    save_npz(BANK_DIR / f"bank_{t}.npz", arr)
RESULT = {"rules": RULES35, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST, "gpu": GPU, "torch": torch.__version__,
          "types_development": TYPES_DEV, "types_evaluation": TYPES_EVAL, "poolings": POOLS, "alphas_grid": [float(ALPHAS[0]), float(ALPHAS[-1]), 0.05],
          "candidates": KEYS, "n_mix": N_MIX, "n_cross_per_type": N_CROSS, "clip_samples": int(NS),
          "counts": {t: {"references": int(len(TR[t])), "test": int(len(TE[t]))} for t in TYPES},
          "G0": {"a": G0["a"]["ok"], "b": bool(all(v["ok"] for v in G0["b"].values())), "c": G0C, "q": {"smallest_cosine": G0Q, "ok": G0Q_OK}, "r": {"per_type": G0R, "ok": G0R_OK}},
          "alpha_star": ALPHA_STAR, "development": DEV, "trainings_read": {f"{ek}|seed {s}": tag for (ek, s), tag in AT_TAGS.items()},
          "seconds": {"data": PREP_S, "passes_this_session": PASS_S, "passes_cell": T_PASSES, "session": round(time.time() - T_START, 1)},
          "evaluation_labels_read": False}
(OUT / "pooling_first_shot.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
ZIP = OUT / "p04_n35_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "pooling_first_shot.json", "pooling_first_shot.json")
    for t in TYPES:
        z.write(BANK_DIR / f"bank_{t}.npz", f"bank/bank_{t}.npz")
print(f"saved pooling_first_shot.json and the bank · zipped → {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB) · session {(time.time() - T_START) / 60:.0f} min")
if IN_COLAB:
    from google.colab import files as _f
    _f.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n35_results.zip`, a few MB). It is also in Drive › `P04_FirstShot35`.